# DL baseline (feedforward neural network)

**GPU recommended -- the first notebook in this suite where it actually helps.** Every earlier baseline (RBAC/ABAC, naive trust, the ML baseline's logistic regression + random forest) is row-wise pandas/sklearn logic with no real matrix math for a GPU to accelerate. This one trains a neural network by gradient descent -- real dense matrix multiplications over mini-batches -- so a GPU runtime (Colab: Runtime -> Change runtime type -> T4 GPU or better) will train noticeably faster. CPU still works (the code picks the device automatically), just slower.

**Same methodology, same features, same target as `05_ml_baseline_variants.ipynb`, on purpose:** this baseline reuses `ml_features.py`'s exact `ALL_FEATURES`/`any_risk` definition, the same train-once-on-`main`/zero-shot-evaluate-on-every-variant protocol, and the same `decision_engine.trust_weighted_decision()` 3-way output as every other baseline. The only thing that changes is the model itself -- a feedforward network (MLP) trained by gradient descent instead of logistic regression / a random forest -- so a difference in the numbers reflects the model family, not a difference in what either one was allowed to see.

**Why not entity embeddings for cpo_id/emsp_id/ev_id:** `ml_features.py`'s own BUG LOG documents why those identity columns are excluded from the classical ML baseline -- the malicious CPO/eMSP are malicious from day 0 with zero time variation, so an identity column (even through a learned embedding instead of a one-hot) lets any model memorize "this specific name is bad" instead of learning the actual behavioral signature. That confound applies to a neural net exactly as much as to logistic regression, so the same features are excluded here, for the same reason -- this stays a fair, directly-comparable baseline, not a leakier one dressed up as "deep learning."

**Requires:** `v2g_benchmark_multi_world.csv` from `02_generate_v2g_benchmark.ipynb` already in your Drive.

## Step 1: check for a GPU

If this prints "No GPU detected," switch the runtime now (Runtime -> Change runtime type -> a GPU, e.g. T4) before continuing -- changing it later would mean re-running every cell from the top anyway, since nothing has been written yet at this point.

In [ ]:
import torch

print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected -- training will run on CPU (slower, but still correct). "
          "Runtime > Change runtime type > GPU (e.g. T4) if you want it faster.")


## Step 2: write the shared modules (decision layer, labels, split, features)

Identical to `05_ml_baseline_variants.ipynb`'s own Step 1 -- `decision_engine.py`, `benchmark_labels.py`, `temporal_split.py`, `ml_features.py` are reused byte-for-byte, so this baseline is judged on exactly the same data/features/target as the classical ML baseline, not a slightly different derivation of "the same" thing.

In [ ]:
%%writefile decision_engine.py
"""
decision_engine.py
-------------------
Implements Algorithm 4 from the TrustMesh framework doc (Section 8):
the trust-weighted three-way access decision (PERMIT / CONDITIONAL-PERMIT / DENY).

This is what turns a trust score into an actual access-control outcome,
which is the concrete instantiation of Gap 4.3.1 (fuzzy trust feeding
MA-ABAC decisions directly, not sitting beside them).
"""

from enum import Enum


class Decision(Enum):
    PERMIT = "PERMIT"
    CONDITIONAL_PERMIT = "CONDITIONAL-PERMIT"
    DENY = "DENY"


def trust_weighted_decision(base_confidence: float, effective_trust: float,
                             theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    """
    effective_confidence = base_confidence * effective_trust   (framework doc Section 8)

    base_confidence: how strongly the attribute itself is asserted (e.g., a
    validly-signed certificate = 1.0; a partially-expired one might be lower).
    effective_trust: E[opinion] = b + a*u for the issuing authority.
    """
    effective_confidence = base_confidence * effective_trust
    if effective_confidence >= theta_high:
        return Decision.PERMIT
    elif effective_confidence >= theta_low:
        return Decision.CONDITIONAL_PERMIT
    else:
        return Decision.DENY


# ---------------------------------------------------------------------------
# Multi-attribute ABAC policy (combines authority_trust with OTHER
# attributes via rules, not just a single trust threshold -- this is what
# makes it attribute-based access control rather than trust-gating alone).
# ---------------------------------------------------------------------------

ROAMING_TRUST_PENALTY = 0.10  # roaming sessions apply a direct penalty to the
                                # CPO's trust value -- a real, common pattern
                                # (foreign/roaming access needs stronger
                                # assurance than home-network access)


def evaluate_access_policy(cpo_trust: float, emsp_trust: float, certificate_valid: bool,
                             is_roaming: bool, is_peak_hour: bool,
                             theta_high: float = 0.75, theta_low: float = 0.4,
                             roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    """
    GENUINE multi-authority ABAC policy: the decision is a function of TWO
    independently-computed authorities' trust scores, not one authority's
    trust plus trust-blind attribute checks.

      - cpo_trust:  TrustMesh's trust in the CPO (billing honesty)
      - emsp_trust: the hub-mediated, EFFECTIVE confidence in the eMSP
                    (certificate-issuance track record), already discounted
                    by how much the relying CPO trusts the hub's reporting
                    (see trust_pipeline.effective_emsp_confidence) -- this is
                    the actual cross-domain fusion mechanism validated in
                    the cold-start experiment, not a flat heuristic.
      - certificate_valid: a per-session, OBJECTIVE fact (hard constraint,
                    e.g. a CRL check) -- NOT subject to either authority's
                    trust score, since it is independently verifiable.

    Combination rule: MIN (weakest-link / AND-conjunction) of the two
    authority trust scores -- both the charging network's billing honesty
    AND the driver's home-network credential issuer must independently
    clear the bar; one being excellent cannot compensate for the other
    being poor. This is what "multi-authority" means operationally: neither
    score alone determines the outcome.
    """
    if not certificate_valid:
        return Decision.DENY  # objective fact, not subject to either authority's trust

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only

    return decision


In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


In [ ]:
%%writefile ml_features.py
"""
ml_features.py
-----------------
Feature set and target shared by the ML and DL baselines. Every input
here is something a live system would actually have at scoring time --
nothing from another baseline's output (RBAC/ABAC decisions, naive
trust scores) and nothing derived from the ground-truth labels
themselves.

TARGET: binary any_risk = (risk_factor_count > 0) -- the same "any real
threat axis active" definition RBAC/ABAC are already judged against
(rbac_baseline.summarize_rbac_blindness, abac_baseline.summarize_single_vs_multi).
Each model's predicted P(any_risk=0) becomes its continuous [0,1] trust
score, fed through decision_engine.trust_weighted_decision() for the
3-way PERMIT/CONDITIONAL-PERMIT/DENY decision -- same as every other
baseline, not reduced to binary.

FEATURES:
  - billing_deviation, latency_deviation: the two fuzzy-engine inputs
    (real-grounded on the metered side, see feature_engineering.py) --
    legitimate raw signals, not another baseline's processed output.
  - certificate_valid, is_roaming, is_peak_hour: real session attributes.
  - previous_flag_rate: ALREADY a real column (enriched_features.py) --
    a causal, prior-only running eMSP dispute rate. Using it here is
    feature engineering, not reusing ABAC's naive_emsp_trust (which is
    just 1 - this same column, renamed for ABAC's own purposes).
  - baseline_charge_duration_h, energy_rate_anomaly: real engineered
    columns from enriched_features.py.
  - charge_duration_h, full_duration_h, idle_h, avg_charge_rate_kw:
    raw session telemetry.
  - ev_session_share_so_far: NEW here -- a causal running statistic (same
    spirit as naive_trust.py's EV computation, RE-DERIVED independently
    so this module has no coupling to the ABAC baseline's internals) --
    the fraction of all sessions seen so far (across the FULL
    chronological sequence, not per-split -- see add_ev_session_share_feature's
    docstring) that belong to this EV.
  - day_idx: raw numeric context.

NOT included, deliberately: ev_id itself (1000-way high-cardinality
identity -- the EV-share feature above is the generalizable signal; the
raw ID would let the model memorize specific train-period EV identities
rather than learn the actual reuse-pattern signature), every
RBAC/ABAC/naive-trust output column (would be benchmarking one baseline
against a feature built from another baseline's output, not an
independent comparison) -- AND, as of this revision, cpo_id/emsp_id too.

BUG LOG (kept here deliberately, not swept away -- same discipline as
fuzzy_trust.py's own bug log): the first version of this module DID
include cpo_id/emsp_id as one-hot categorical features. On the real
125,910-row benchmark this produced AUROC/AUPRC = 1.0000 EXACTLY for
both logistic regression and random forest -- not a good score, a tell.
Root cause: in this dataset the malicious CPO/eMSP are malicious from
day 0 with no time variation (onset_day=0), so cpo_id=="CPO-B" implies
gt_cpo_malicious=1 with zero exceptions, same for emsp_id=="eMSP-C" --
a model with those IDs as input doesn't need billing_deviation or
previous_flag_rate at all; it just memorizes "if name is CPO-B or
eMSP-C, flag it," which is a lookup table, not a behavioral detector,
and wouldn't generalize to a deployment with different CPO/eMSP names.
(A synthetic smoke test run before the real data was available had the
same confound baked into its own generator, which is why it showed a
suspiciously-high-but-not-quite-1.0 AUROC~0.995 instead of flagging the
problem outright -- the real data's exact 1.0000 was the actual tell.)
Fixed by dropping cpo_id/emsp_id entirely: every remaining feature is a
genuine behavioral/contextual signal (billing_deviation,
previous_flag_rate, ev_session_share_so_far, etc.), so whatever
separability the model achieves now reflects learning from behavior,
not memorizing which specific name is the known bad one -- the same
standard TrustMesh itself is held to (its trust score comes from
billing_deviation observations, not from knowing a name in advance).

SECOND BUG LOG ENTRY (found while wiring up the 7-build variant suite,
confirmed PRE-EXISTING -- present even for the original single-build,
2-world "main" dataset): add_ev_session_share_feature's running count
and its denominator (total_seen) were kept across the ENTIRE passed-in
frame with no awareness of which world or variant a row came from --
the same class of cross-world leak documented in naive_trust.py's own
BUG LOG (world 0 and world 1, and now all 7 variants, share the same
real connection_time values, so sorting the union together interleaves
independent synthetic universes instead of treating each as its own
clean timeline). Fixed the same way: grouped by (variant, world_id)
before computing the running share, with a no-grouping fallback when
those columns aren't present."""

import numpy as np
import pandas as pd

NUMERIC_FEATURES = [
    "billing_deviation", "latency_deviation", "previous_flag_rate",
    "baseline_charge_duration_h", "energy_rate_anomaly",
    "charge_duration_h", "full_duration_h", "idle_h", "avg_charge_rate_kw",
    "ev_session_share_so_far", "day_idx",
]
BOOLEAN_FEATURES = ["certificate_valid", "is_roaming", "is_peak_hour"]
ALL_FEATURES = NUMERIC_FEATURES + BOOLEAN_FEATURES
TARGET = "any_risk"


DEFAULT_GROUP_COLS = ("variant", "world_id")  # see SECOND BUG LOG entry above


def add_ev_session_share_feature(df: pd.DataFrame, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """Causal running per-EV session share. MUST be called on the FULL
    chronological sequence (train+test combined) BEFORE any train/test
    split -- calling it separately on an already-split test frame would
    reset every EV's running count to zero at the split boundary,
    understating exactly the reuse signature this feature exists to
    capture. (Independent of naive_trust.py's analogous computation,
    deliberately re-derived here so the ML feature set has no coupling
    to the ABAC baseline's internals.)

    Computed independently within each (variant, world_id) group when
    those columns are present (falls back to the whole frame as one
    group otherwise) -- see this module's SECOND BUG LOG entry."""
    df = df.copy()
    present = [c for c in group_cols if c in df.columns]
    groups = df.groupby(present, sort=False) if present else [(None, df)]

    share_col = pd.Series(np.nan, index=df.index, dtype=float)
    for _, g in groups:
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            local[i] = (this_ev_n / total_seen) if total_seen > 0 else 0.0
            running_n[e] = this_ev_n + 1
            total_seen += 1
        share_col.loc[order] = local

    df["ev_session_share_so_far"] = share_col
    return df


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET] = (df["risk_factor_count"] > 0).astype(int)
    return df


def prepare_ml_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Adds the EV-share feature and target, and coerces the boolean
    columns to int (sklearn-friendly). Call on the FULL combined
    chronological sequence, before splitting -- see
    add_ev_session_share_feature's docstring."""
    df = add_ev_session_share_feature(df)
    df = add_target(df)
    for col in BOOLEAN_FEATURES:
        df[col] = df[col].astype(int)
    return df


## Step 3: clear stale checkpoints, then write and train the DL baseline

Same unconditional-clear discipline as `05`'s own Step 2 -- the output folder is wiped before training so a re-run of this notebook never silently reuses a checkpoint from a previous architecture/feature change. `dl_baseline.py`'s own docstring has the full design rationale (why these features, why no entity embeddings, why GPU matters here).

In [ ]:
import os
import shutil, os
dl_out_dir = os.environ.get('V2G_DL_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/dl_baseline')
if os.path.isdir(dl_out_dir):
    shutil.rmtree(dl_out_dir)
    print(f'Cleared {dl_out_dir}')
else:
    print('Nothing to clear.')


In [ ]:
%%writefile dl_baseline.py
"""
dl_baseline.py
----------------
Deep-learning baseline: a feedforward neural network (PyTorch MLP) on the
EXACT SAME feature set and target as the classical ML baseline
(ml_baseline.py / ml_features.ALL_FEATURES / "any_risk") -- same
train-on-'main'-only, zero-shot-eval-on-every-variant methodology, same
decision_engine.trust_weighted_decision() 3-way output, same
world_id-keyed predictions-CSV convention as every other baseline. The
only thing that changes relative to ml_baseline.py is the model itself:
a gradient-descent-trained feedforward network instead of logistic
regression / a random forest -- so a difference in the results reflects
the model family, not a difference in what either one was allowed to see.

WHY THE SAME FEATURES, NOT ENTITY EMBEDDINGS: ml_features.py's own BUG LOG
documents why cpo_id/emsp_id/ev_id are deliberately excluded -- in this
dataset the malicious CPO/eMSP are malicious from day 0 with no time
variation, so including their IDENTITY (even through a learned embedding
instead of a one-hot column) would let any model memorize "this specific
name is bad" instead of learning the underlying behavioral signature
(billing_deviation, previous_flag_rate, ev_session_share_so_far, ...).
That confound applies to a neural net exactly as much as to logistic
regression, so it is excluded here for the same reason -- this stays a
fair, directly-comparable baseline, not a leakier one dressed up as
"deep learning."

GPU: the first baseline in this suite where a GPU runtime actually helps
-- real gradient descent over dense matrix multiplications, unlike every
earlier baseline's row-wise pandas/sklearn/rule logic. Device is picked
automatically (_device()); runs on CPU too, just slower, if no GPU
runtime is selected.

VALIDATION SPLIT FOR EARLY STOPPING: carved from the END of 'main''s
TRAIN days (VAL_FRACTION_OF_TRAIN_DAYS), not a random shuffle -- same
temporal-not-random discipline as temporal_split.py's own train/test
cut (see that module's docstring for why: random shuffling would let
adjacent-day rows leak across the boundary). The test split itself is
untouched by this -- it's only used to decide when to stop training.

CHECKPOINTING: the trained weights (state_dict, .pt) and the fitted
StandardScaler (.joblib, needed to preprocess new rows identically at
eval time) are saved together the moment training finishes; re-running
skips training (loads both) if the checkpoint already exists -- same
discipline as ml_baseline.py's joblib checkpoint.
"""

import os
import sys
import time

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, os.path.dirname(__file__) or ".")

import ml_features
import benchmark_labels
import temporal_split
from decision_engine import trust_weighted_decision

OUT_DIR = os.environ.get(
    "V2G_DL_OUT_DIR",
    "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/dl_baseline",
)

TRAIN_VARIANT = "main"  # train exclusively on this build; every other build
                          # is evaluated zero-shot -- mirrors ml_baseline.py
MODEL_NAME = "mlp"
SEED = 42

VAL_FRACTION_OF_TRAIN_DAYS = 0.15  # carved from the end of main's train days
HIDDEN_SIZES = (64, 32)
DROPOUT = 0.2
LR = 1e-3
MAX_EPOCHS = 60
PATIENCE = 8          # early stop if val AUROC hasn't improved for this many epochs
BATCH_SIZE = 2048


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    """Plain feedforward network: Linear -> ReLU -> Dropout, stacked per
    HIDDEN_SIZES, ending in a single logit (BCEWithLogitsLoss applies the
    sigmoid internally, so no activation on the output layer)."""

    def __init__(self, n_features, hidden_sizes=HIDDEN_SIZES, dropout=DROPOUT):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def model_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}.pt")


def scaler_checkpoint_path():
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_scaler.joblib")


def variant_preds_path(variant_name):
    return os.path.join(OUT_DIR, f"{MODEL_NAME}_{variant_name}_test_predictions.csv")


def _prepare_full_frame(combined_csv_path):
    """Identical to ml_baseline.py's _prepare_full_frame -- same frame, same
    feature computation -- so this baseline and the classical ML one are
    judged on EXACTLY the same data, not two slightly different derivations
    of "the same" data."""
    df = pd.read_csv(combined_csv_path)
    df["connection_time"] = pd.to_datetime(df["connection_time"], utc=True)
    df = benchmark_labels.add_ground_truth_labels(df)
    df = ml_features.prepare_ml_frame(df)
    day_cutoff = temporal_split.compute_day_cutoff(df)
    return df, day_cutoff


def _train_val_day_cutoff(train_df, val_fraction=VAL_FRACTION_OF_TRAIN_DAYS):
    days = np.sort(train_df["day_idx"].unique())
    cutoff_idx = int(len(days) * (1 - val_fraction))
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both non-empty
    return int(days[cutoff_idx])


def get_or_train_model(df, day_cutoff):
    """Loads the checkpoint (model + scaler) if one already exists;
    otherwise trains on TRAIN_VARIANT's ("main") train split only (minus
    a tail-of-days validation slice for early stopping) and saves both.
    Mirrors ml_baseline.py's get_or_train_model."""
    model_path = model_checkpoint_path()
    scaler_path = scaler_checkpoint_path()
    device = _device()
    n_features = len(ml_features.ALL_FEATURES)
    model = MLP(n_features).to(device)

    if os.path.exists(model_path) and os.path.exists(scaler_path):
        print(f"[{MODEL_NAME}] checkpoint already exists -- loading, not retraining (device={device})")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        scaler = joblib.load(scaler_path)
        return model, scaler, device

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    full_train_df = df[(df["variant"] == TRAIN_VARIANT) & (df["day_idx"] < day_cutoff)]
    val_cutoff = _train_val_day_cutoff(full_train_df)
    train_df = full_train_df[full_train_df["day_idx"] < val_cutoff]
    val_df = full_train_df[full_train_df["day_idx"] >= val_cutoff]

    scaler = StandardScaler().fit(train_df[ml_features.ALL_FEATURES])
    X_train = scaler.transform(train_df[ml_features.ALL_FEATURES])
    X_val = scaler.transform(val_df[ml_features.ALL_FEATURES])
    y_train = train_df[ml_features.TARGET].to_numpy().astype(np.float32)
    y_val = val_df[ml_features.TARGET].to_numpy().astype(np.float32)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)

    # pos_weight mirrors LR/RF's class_weight="balanced" -- any_risk is
    # roughly 59/41 here, not severely imbalanced, but kept for parity
    # with the other two baselines' documented choice.
    n_pos = float(y_train.sum())
    n_neg = float(len(y_train) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_auroc = -1.0
    best_state = None
    epochs_without_improvement = 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t).cpu().numpy()
        val_proba = 1.0 / (1.0 + np.exp(-val_logits))
        val_auroc = roc_auc_score(y_val, val_proba)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0 or epochs_without_improvement >= PATIENCE:
            print(f"  epoch {epoch:3d}  val_auroc={val_auroc:.4f}  best={best_val_auroc:.4f}  "
                  f"no_improve={epochs_without_improvement}")

        if epochs_without_improvement >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (no val AUROC improvement for {PATIENCE} epochs)")
            break

    model.load_state_dict(best_state)
    model.eval()

    os.makedirs(OUT_DIR, exist_ok=True)
    torch.save(model.state_dict(), model_path)
    joblib.dump(scaler, scaler_path)
    print(f"[{MODEL_NAME}] trained on '{TRAIN_VARIANT}' only: n_train={len(train_df)}  n_val={len(val_df)}  "
          f"best_val_auroc={best_val_auroc:.4f}  ({time.time() - t0:.1f}s, device={device}) -> {model_path}")
    return model, scaler, device


def evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name):
    """Scores the ALREADY-TRAINED (on 'main' only) network against
    variant_name's OWN test split -- zero-shot, no retraining. Mirrors
    ml_baseline.py's evaluate_on_variant, including why world_id is saved
    (session_id alone is not a unique join key -- see that module's own
    comment for the full account of the bug this avoids)."""
    test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)]
    X_test = scaler.transform(test_df[ml_features.ALL_FEATURES])
    y_test = test_df[ml_features.TARGET].to_numpy()

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
    proba_risk = 1.0 / (1.0 + np.exp(-logits))
    trust_score = 1.0 - proba_risk

    auroc = roc_auc_score(y_test, proba_risk)
    auprc = average_precision_score(y_test, proba_risk)
    decisions = [trust_weighted_decision(1.0, float(t)).value for t in trust_score]

    preds_path = variant_preds_path(variant_name)
    out = test_df[["session_id", "variant", "world_id", "cpo_id", "emsp_id", "ev_id", "day_idx",
                    "risk_factor_count", "any_risk"]].copy()
    out["trust_score"] = trust_score
    out["decision"] = decisions
    out.to_csv(preds_path, index=False)

    return {"model": MODEL_NAME, "variant": variant_name,
            "auroc": float(auroc), "auprc": float(auprc), "n_test": int(len(test_df))}


def main(combined_csv_path):
    df, day_cutoff = _prepare_full_frame(combined_csv_path)
    variants = sorted(df["variant"].unique()) if "variant" in df.columns else [None]
    device = _device()
    print(f"day_cutoff={day_cutoff}  variants found: {variants}")
    print(f"Training 1 model ({MODEL_NAME}) on '{TRAIN_VARIANT}' only, then evaluating "
          f"zero-shot on every variant's own test split (device={device})")

    model, scaler, device = get_or_train_model(df, day_cutoff)

    all_rows = []
    for variant_name in variants:
        row = evaluate_on_variant(model, scaler, device, df, day_cutoff, variant_name)
        all_rows.append(row)
        print(f"  [{MODEL_NAME:10s} -> {variant_name:16s}] "
              f"AUROC={row['auroc']:.4f}  AUPRC={row['auprc']:.4f}  n_test={row['n_test']}")

    summary = pd.DataFrame(all_rows)
    summary_path = os.path.join(OUT_DIR, "dl_baseline_variant_summary.csv")
    os.makedirs(OUT_DIR, exist_ok=True)
    summary.to_csv(summary_path, index=False)

    print(f"\n=== Summary table (trained once on '{TRAIN_VARIANT}', zero-shot evaluated "
          f"per variant) -- saved to {summary_path} ===")
    print(summary.set_index("variant")[["auroc", "auprc"]].round(4).to_string())

    seed_variants = [v for v in variants if v.startswith("seed_")]
    if seed_variants:
        print(f"\n=== Stability check across {len(seed_variants)} independent reseeds "
              f"({', '.join(seed_variants)}) -- mean +/- std AUROC ===")
        stab = summary[summary["variant"].isin(seed_variants)]["auroc"]
        print(f"  mean={stab.mean():.4f}  std={stab.std():.4f}")

    return summary


if __name__ == "__main__":
    combined_csv_path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    main(combined_csv_path)


In [ ]:
import os
import dl_baseline

combined_path = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
summary = dl_baseline.main(combined_path)


## Step 4: insights -- AUROC per variant, and stability across seeds

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DL_OUT_DIR = os.environ.get('V2G_DL_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/dl_baseline')
summary = pd.read_csv(os.path.join(DL_OUT_DIR, 'dl_baseline_variant_summary.csv'))
summary = summary.set_index('variant')
print(summary[['auroc', 'auprc']].round(4))

ax = summary['auroc'].plot(kind='bar', figsize=(9, 5), ylim=(0, 1.05), legend=False, color='#5470c6')
ax.set_ylabel('AUROC (test split, zero-shot per variant)')
ax.set_title("DL baseline (MLP): trained once on 'main', evaluated zero-shot per variant")
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('dl_baseline_variant_insights.pdf', dpi=300)
plt.show()
print('Saved dl_baseline_variant_insights.pdf')

seed_rows = summary[summary.index.str.startswith('seed_')]
if len(seed_rows):
    print(f"\nStability across {len(seed_rows)} independent reseeds -- "
          f"mean={seed_rows['auroc'].mean():.4f}  std={seed_rows['auroc'].std():.4f}")


## Step 5: F1/precision/recall/confusion matrix, per-axis AUROC, duplicate-key sanity check

Same `(session_id, variant, world_id)` join-key discipline as `05`'s own Step 4/5 (see that notebook's intro for the full account of the session_id-reuse bug this avoids) -- the assert below is load-bearing, not decorative. Unlike `05`'s delivered copy, there's no `drop_duplicates` band-aid here: the underlying duplicate-raw-row issue is fixed at the source in `raw_preprocess.py` (notebook 02), so this assert is expected to pass cleanly on its own. A quick duplicate-key count is printed first as a direct, standing check of that.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              average_precision_score, brier_score_loss)
import benchmark_labels
import glob, os as _os

OUT_DIR = os.environ.get('V2G_DL_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/dl_baseline')
COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
MODEL_NAME = 'mlp'
DECISION_THRESHOLD = 0.5
JOIN_KEYS = ['session_id', 'variant', 'world_id']

full_labels = pd.read_csv(COMBINED_CSV)
full_labels = benchmark_labels.add_ground_truth_labels(full_labels)

dup_count = full_labels.duplicated(subset=JOIN_KEYS).sum()
print(f'duplicate rows on {JOIN_KEYS}: {dup_count} (expect 0 -- see this cell\'s markdown)')
assert dup_count == 0, f'{dup_count} duplicate rows on {JOIN_KEYS} in the combined CSV -- stop and investigate'
labels_df = full_labels[JOIN_KEYS + ['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']]

variants = sorted({
    _os.path.basename(p)[len(f'{MODEL_NAME}_'):-len('_test_predictions.csv')]
    for p in glob.glob(_os.path.join(OUT_DIR, f'{MODEL_NAME}_*_test_predictions.csv'))
})
print(f'variants found from saved prediction files: {variants}\n')

overall_rows, axis_rows = [], []
for variant in variants:
    preds = pd.read_csv(f'{OUT_DIR}/{MODEL_NAME}_{variant}_test_predictions.csv')
    assert 'world_id' in preds.columns, f'{variant}: predictions file has no world_id column'
    n_before = len(preds)
    preds = preds.merge(labels_df, on=JOIN_KEYS, how='left')
    assert len(preds) == n_before, f'{variant}: join fanned out ({n_before} -> {len(preds)})'

    y_true = preds['any_risk'].astype(int).to_numpy()
    proba_risk = 1.0 - preds['trust_score'].to_numpy()
    y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
    flagged_by_decision = preds['decision'].isin(['DENY', 'CONDITIONAL-PERMIT']).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall_rows.append({
        'variant': variant,
        'auroc': roc_auc_score(y_true, proba_risk),
        'auprc': average_precision_score(y_true, proba_risk),
        'brier': brier_score_loss(y_true, proba_risk),
        'f1@0.5': f1_score(y_true, y_pred, zero_division=0),
        'precision@0.5': precision_score(y_true, y_pred, zero_division=0),
        'recall@0.5': recall_score(y_true, y_pred, zero_division=0),
        'accuracy@0.5': accuracy_score(y_true, y_pred),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'f1_decision': f1_score(y_true, flagged_by_decision, zero_division=0),
    })

    for axis in ['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']:
        y_axis = preds[axis].astype(int).to_numpy()
        axis_auroc = float('nan') if (y_axis.sum() == 0 or y_axis.sum() == len(y_axis)) else roc_auc_score(y_axis, proba_risk)
        axis_rows.append({'variant': variant, 'axis': axis, 'auroc': axis_auroc})

overall = pd.DataFrame(overall_rows)
by_axis = pd.DataFrame(axis_rows)
metrics_path = f'{OUT_DIR}/dl_baseline_variant_metrics_full.csv'
overall.to_csv(metrics_path, index=False)
print(f'Saved -> {metrics_path}\n')

print('=== overall (any_risk) metrics ===')
print(overall.set_index('variant')[
    ['auroc', 'auprc', 'brier', 'f1@0.5', 'precision@0.5', 'recall@0.5', 'accuracy@0.5', 'f1_decision']
].round(4).to_string())
print()
print('=== confusion matrix (tn/fp/fn/tp @ proba>=0.5) ===')
print(overall.set_index('variant')[['tn', 'fp', 'fn', 'tp']].to_string())
print()
print('=== per-axis AUROC (does the SAME score separate each individual threat axis?) ===')
print(by_axis.pivot(index='variant', columns='axis', values='auroc').round(4).to_string())
print()

for variant in variants:
    preds = pd.read_csv(f'{OUT_DIR}/{MODEL_NAME}_{variant}_test_predictions.csv')
    y_true = preds['any_risk'].astype(int).to_numpy()
    proba_risk = 1.0 - preds['trust_score'].to_numpy()
    y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
    print(f'--- {variant} ---')
    print(classification_report(y_true, y_pred, target_names=['no_risk', 'any_risk'], digits=4, zero_division=0))


## Done

Output: `dl_baseline_variant_summary.csv` (AUROC/AUPRC per variant) and `dl_baseline_variant_metrics_full.csv` (the fuller F1/precision/recall/confusion-matrix/per-axis table) in `.../worlds/dl_baseline/`, plus the trained network (`mlp.pt`) and its fitted scaler (`mlp_scaler.joblib`) so re-running this notebook doesn't retrain from scratch.

Paste back: the training log (epochs + early-stopping point, and which device it trained on), the per-variant AUROC/AUPRC table, the stability check across the 3 reseeds, and the Step 4 metrics table -- directly comparable to `05`'s own tables, same features/target/protocol, different model.